# 07. Post-hoc Gene-Space Evaluation

This notebook reconstructs gene-space predictions from the trained latent model and computes biologically interpretable diagnostics. The workflow mirrors the current post-analysis logic: it reloads the best checkpoint and the train-only PCA model, performs inference on test and OOD subsets, inverse-transforms predictions into gene space, and evaluates DEG recovery, pathway-level pseudobulk profile recovery, and dose-trend agreement. Every result is regenerated from disk so that no hidden notebook state is required.


In [1]:
!pip -q install anndata scanpy scikit-learn scipy pandas numpy torch pyarrow

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import urllib.request
from pathlib import Path

HELPER_DIR = Path("/content/drive/MyDrive/ChemDFM")
if str(HELPER_DIR) not in sys.path:
    sys.path.append(str(HELPER_DIR))

from chemdfm_notebook_helpers import *

DATA_PATH = Path("/content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    print("Downloading SciPlex dataset...")
    URL = "https://f003.backblazeb2.com/file/chemCPA-datasets/sciplex_complete_middle_subset.h5ad"
    urllib.request.urlretrieve(URL, DATA_PATH)
    print("Download complete.")

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_PATH =", DATA_PATH)
print("Using device:", DEVICE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 16.7 MB/s eta 0:00:00
Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/ChemDFM
DATA_PATH = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad
Using device: cpu


In [2]:
RUN_NAME = "residual_cellaware_mmd"
RUN_DIR = RUNS_DIR / RUN_NAME
BEST_CKPT = RUN_DIR / "checkpoints" / "best_residual_cellaware_mmd.pt"
assert BEST_CKPT.exists(), BEST_CKPT

@dataclass
class CFG:
    split_col: str = "split_ho_pathway"
    pca_dim: int = 25
    batch_size: int = 512
    emb_dim: int = 32
    hidden: int = 256
    dose_hidden: int = 32
    dropout: float = 0.1
    num_workers: int = 0
    pin_memory: bool = False
    deg_topk: int = 50
    dose_min_unique: int = 3
cfg = CFG()


In [4]:
adata, X = load_adata(split_col=cfg.split_col)
obs = pd.read_parquet(PROCESSED_DIR / "datasets" / "adata_obs_processed.parquet")
adata.obs = obs.copy()

with open(PROCESSED_DIR / "pca" / f"pca_model_{cfg.pca_dim}d.pkl", "rb") as f:
    pca = pickle.load(f)
with open(PROCESSED_DIR / "encoders" / "drug_encoder.pkl", "rb") as f:
    drug_enc = pickle.load(f)
with open(PROCESSED_DIR / "encoders" / "cell_encoder.pkl", "rb") as f:
    cell_enc = pickle.load(f)

train_mask = (adata.obs["_split3"] == "train").values
X_pca = np.zeros((adata.n_obs, cfg.pca_dim), dtype=np.float32)
X_pca[train_mask] = pca.transform(X[train_mask]).astype(np.float32)
X_pca[~train_mask] = pca.transform(X[~train_mask]).astype(np.float32)

control_mask = (adata.obs["condition"].astype(str) == "control").values
ctrl_means_pca, ctrl_means_gene = {}, {}
for cell in sorted(adata.obs["cell_type"].astype(str).unique()):
    m = control_mask & (adata.obs["cell_type"].astype(str).values == cell)
    if m.sum() == 0:
        raise ValueError(f"No control cells for {cell}")
    ctrl_means_pca[cell] = X_pca[m].mean(axis=0).astype(np.float32)
    ctrl_means_gene[cell] = X[m].mean(axis=0).astype(np.float32)

X0_pca = np.stack([ctrl_means_pca[c] for c in adata.obs["cell_type"].astype(str).values]).astype(np.float32)
X0_gene = np.stack([ctrl_means_gene[c] for c in adata.obs["cell_type"].astype(str).values]).astype(np.float32)

test_ds = ChemResidualDataset(adata, X_pca, X0_pca, split="test", include_idx=True, require_pathway=True)
ood_ds = ChemResidualDataset(adata, X_pca, X0_pca, split="ood", include_idx=True, require_pathway=True)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
ood_loader = DataLoader(ood_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [5]:
model = ResidualDoseResponseModel(
    latent_dim=cfg.pca_dim,
    n_drugs=len(drug_enc.classes_),
    n_cells=len(cell_enc.classes_),
    emb_dim=cfg.emb_dim,
    hidden=cfg.hidden,
    dose_hidden=cfg.dose_hidden,
    dropout=cfg.dropout,
).to(DEVICE)
model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE))
model.eval()

@torch.no_grad()
def infer_split(loader, split_name):
    rows = []
    pred_pca_all, true_pca_all, x0_pca_all = [], [], []
    pred_gene_all, true_gene_all, x0_gene_all = [], [], []
    for batch in loader:
        idxs = np.array(batch["idx"], dtype=int)
        x0 = batch["x0"].to(DEVICE)
        x_true = batch["x_true"].to(DEVICE)
        drug_idx = batch["drug_idx"].to(DEVICE)
        cell_idx = batch["cell_idx"].to(DEVICE)
        dose = batch["dose"].to(DEVICE)

        _, x_hat = model(x0, drug_idx, cell_idx, dose)
        x_hat_np = x_hat.cpu().numpy()
        x_true_np = x_true.cpu().numpy()
        x0_np = x0.cpu().numpy()

        xhat_gene = pca.inverse_transform(x_hat_np).astype(np.float32)
        xtrue_gene = X[idxs].astype(np.float32)
        x0gene = X0_gene[idxs].astype(np.float32)

        pred_pca_all.append(x_hat_np); true_pca_all.append(x_true_np); x0_pca_all.append(x0_np)
        pred_gene_all.append(xhat_gene); true_gene_all.append(xtrue_gene); x0_gene_all.append(x0gene)

        for idx in idxs:
            obs_row = adata.obs.iloc[idx]
            rows.append({
                "idx": int(idx),
                "split": split_name,
                "cell_type": str(obs_row["cell_type"]),
                "condition": str(obs_row["condition"]),
                "pathway": str(obs_row["pathway"]),
                "dose": float(obs_row["dose"]),
            })

    return (
        pd.DataFrame(rows),
        np.concatenate(pred_pca_all),
        np.concatenate(true_pca_all),
        np.concatenate(x0_pca_all),
        np.concatenate(pred_gene_all),
        np.concatenate(true_gene_all),
        np.concatenate(x0_gene_all),
    )

test_meta, test_pred_pca, test_true_pca, test_x0_pca, test_pred_gene, test_true_gene, test_x0_gene = infer_split(test_loader, "test")
ood_meta, ood_pred_pca, ood_true_pca, ood_x0_pca, ood_pred_gene, ood_true_gene, ood_x0_gene = infer_split(ood_loader, "ood")


In [6]:
def compute_deg_auprc(meta, pred_gene, true_gene, x0_gene, topk=50):
    rows = []
    true_delta = true_gene - x0_gene
    pred_delta = pred_gene - x0_gene
    for i in range(true_gene.shape[0]):
        true_abs = np.abs(true_delta[i])
        pred_abs = np.abs(pred_delta[i])
        y_true = np.zeros_like(true_abs, dtype=np.int32)
        top_idx = np.argsort(-true_abs)[:topk]
        y_true[top_idx] = 1
        if y_true.sum() == 0 or y_true.sum() == len(y_true):
            continue
        auprc = average_precision_score(y_true, pred_abs)
        rows.append({
            "split": meta.iloc[i]["split"],
            "cell_type": meta.iloc[i]["cell_type"],
            "condition": meta.iloc[i]["condition"],
            "pathway": meta.iloc[i]["pathway"],
            "dose": meta.iloc[i]["dose"],
            "deg_auprc": float(auprc),
        })
    return pd.DataFrame(rows)

def pathway_recovery(meta, pred_gene, true_gene):
    tmp = meta.copy()
    tmp["row_idx"] = np.arange(len(tmp))
    rows = []
    for keys, sub in tmp.groupby(["split", "cell_type", "pathway"]):
        idx = sub["row_idx"].values
        if len(idx) < 5:
            continue
        pred_mean = pred_gene[idx].mean(axis=0)
        true_mean = true_gene[idx].mean(axis=0)
        corr = np.nan if (np.std(pred_mean) < 1e-8 or np.std(true_mean) < 1e-8) else pearsonr(pred_mean, true_mean)[0]
        rows.append({
            "split": keys[0],
            "cell_type": keys[1],
            "pathway": keys[2],
            "n_samples": int(len(idx)),
            "pathway_profile_corr": float(corr) if corr == corr else np.nan,
            "pathway_profile_r2": float(r2_score(true_mean, pred_mean)),
        })
    return pd.DataFrame(rows)

def dose_trend_diag(meta, pred_gene, true_gene, x0_gene, min_unique=3):
    tmp = meta.copy()
    tmp["row_idx"] = np.arange(len(tmp))
    rows = []
    for keys, sub in tmp.groupby(["split", "cell_type", "condition"]):
        unique_doses = np.sort(sub["dose"].unique())
        if len(unique_doses) < min_unique:
            continue
        dose_vals, true_mags, pred_mags = [], [], []
        for d in unique_doses:
            s = sub[sub["dose"] == d]
            idx = s["row_idx"].values
            pred_delta = pred_gene[idx] - x0_gene[idx]
            true_delta = true_gene[idx] - x0_gene[idx]
            pred_mag = np.mean(np.linalg.norm(pred_delta, axis=1))
            true_mag = np.mean(np.linalg.norm(true_delta, axis=1))
            dose_vals.append(float(d)); pred_mags.append(float(pred_mag)); true_mags.append(float(true_mag))
        rho = np.nan if (len(set(true_mags)) < 2 or len(set(pred_mags)) < 2) else spearmanr(true_mags, pred_mags).correlation
        rho_true_dose = spearmanr(dose_vals, true_mags).correlation if len(set(true_mags)) >= 2 else np.nan
        rho_pred_dose = spearmanr(dose_vals, pred_mags).correlation if len(set(pred_mags)) >= 2 else np.nan
        rows.append({
            "split": keys[0], "cell_type": keys[1], "condition": keys[2], "n_doses": len(unique_doses),
            "trend_match_spearman": float(rho) if rho == rho else np.nan,
            "true_dose_monotonicity": float(rho_true_dose) if rho_true_dose == rho_true_dose else np.nan,
            "pred_dose_monotonicity": float(rho_pred_dose) if rho_pred_dose == rho_pred_dose else np.nan,
        })
    return pd.DataFrame(rows)


In [7]:
final_overall = pd.DataFrame([
    {"split": "test", **compute_metrics(test_pred_pca, test_true_pca, test_x0_pca)},
    {"split": "ood", **compute_metrics(ood_pred_pca, ood_true_pca, ood_x0_pca)},
])

final_per_cell_rows = []
for split_name, meta, pred, true, x0 in [
    ("test", test_meta, test_pred_pca, test_true_pca, test_x0_pca),
    ("ood", ood_meta, ood_pred_pca, ood_true_pca, ood_x0_pca),
]:
    for cell in sorted(meta["cell_type"].unique()):
        m = meta["cell_type"].values == cell
        final_per_cell_rows.append({"split": split_name, "cell_type": cell, **compute_metrics(pred[m], true[m], x0[m])})
final_per_cell = pd.DataFrame(final_per_cell_rows)

deg_df = pd.concat([
    compute_deg_auprc(test_meta, test_pred_gene, test_true_gene, test_x0_gene, topk=cfg.deg_topk),
    compute_deg_auprc(ood_meta, ood_pred_gene, ood_true_gene, ood_x0_gene, topk=cfg.deg_topk),
], ignore_index=True)
deg_summary = deg_df.groupby(["split", "cell_type"], as_index=False)["deg_auprc"].mean()

pathway_df = pd.concat([
    pathway_recovery(test_meta, test_pred_gene, test_true_gene),
    pathway_recovery(ood_meta, ood_pred_gene, ood_true_gene),
], ignore_index=True)
pathway_summary = pathway_df.groupby(["split", "cell_type"], as_index=False)[["pathway_profile_corr", "pathway_profile_r2"]].mean()

dose_df = pd.concat([
    dose_trend_diag(test_meta, test_pred_gene, test_true_gene, test_x0_gene, min_unique=cfg.dose_min_unique),
    dose_trend_diag(ood_meta, ood_pred_gene, ood_true_gene, ood_x0_gene, min_unique=cfg.dose_min_unique),
], ignore_index=True)
dose_summary = dose_df.groupby(["split", "cell_type"], as_index=False)[["trend_match_spearman", "true_dose_monotonicity", "pred_dose_monotonicity"]].mean()

OUT = RESULTS_DIR / "canonical"
OUT.mkdir(parents=True, exist_ok=True)
final_overall.to_csv(OUT / "final_overall_metrics_posthoc.csv", index=False)
final_per_cell.to_csv(OUT / "final_per_cell_metrics_posthoc.csv", index=False)
deg_df.to_csv(OUT / "deg_auprc_per_sample.csv", index=False)
deg_summary.to_csv(OUT / "deg_auprc_summary.csv", index=False)
pathway_df.to_csv(OUT / "pathway_recovery_groupwise.csv", index=False)
pathway_summary.to_csv(OUT / "pathway_recovery_summary.csv", index=False)
dose_df.to_csv(OUT / "dose_trend_groupwise.csv", index=False)
dose_summary.to_csv(OUT / "dose_trend_summary.csv", index=False)

final_overall


,split,r2_full,pearson_rowmean,mse,collapse_ratio,mean_shift_error,r2_top20,r2_top50
0,test,0.635105,0.783744,0.342062,0.647115,0.442125,0.498626,0.575433
1,ood,0.615376,0.772934,0.358428,0.596358,0.453690,0.486698,0.558478


The biological evaluation now exists as a fully disk-based notebook. The following notebook extends this internal biological validation by comparing true and predicted gene rankings against external pathway resources such as GO, Reactome, KEGG, or MSigDB when those resources are available locally.
